## Gemini Model

A workflow that uses Google's Gemini LLM to search for novel and creative implementations of the NMF algorithm.

Initially will focus on improvements to the LS-NMF algorithm, then will search for improvements to the WS-NMF algorithm.

The workflow contains the following steps:
1. Prompt Gemini for a better implementation of the base algorithm, passing in the base model as text.
   1. The prompt will request n new models. 
2. Parse the results to extract the n 'new' versions of the algorithm.
3. Create a new python function for each of the algorithm.
4. Validate each of the new algorithms by creating a new BatchNMF model, and manually swapping out the self.update_step function with the new function.
5. Run 20 models, aggregate the results (mean runtime, min/mean/max Q(true), min/mean/max Q(robust))
6. Save the top 2 models to mongodb, along with aggregated results.
   1. Any number could be saved, but 2 provides options for the selection process.
7. Repeat steps 1-6 using a random selection from the new algorithm database.

In [ ]:
import os
import sys
import pathlib
import textwrap
import inspect

import google.generativeai as genai

from Isrc.display import display
from Isrc.display import Markdown

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
from esat.data.datahandler import DataHandler
from esat.model.nmf import NMF
from esat.model.batch_nmf import BatchNMF

## Base LS-NMF update algorithm
from esat.model.ls_nmf import LSNMF
base_alg = inspect.getsourcelines(LSNMF.update)
base_code = []
for i in range(len(base_alg[0])):
    if i > 0 and i < 7 or i >= 30:
        code_line = base_alg[0][i]
        if "def" in code_line:
            code_line = textwrap.dedent(code_line)
            if "self" not in code_line:
                code_line = code_line.replace("\n", "self,\n")
        base_code.append(code_line)        
# base_code = "".join(base_alg[0][1:7]) + "".join(base_alg[0][30:])
base_code = "".join(base_code)
# base_code = base_code[0: -1]
# base_code = str(base_code).replace("[", "")
# base_code = str(base_code).replace("]", "")
base_header = base_alg[0][1:7]
base_return = base_alg[0][-1:]
base_code

In [ ]:
## Base WS-NMF update algorithm
from esat.model.ws_nmf import WSNMF
ws_base_alg = inspect.getsourcelines(WSNMF.update)
ws_base_code = []
for i in range(1, len(ws_base_alg[0])):
    ws_code_line = ws_base_alg[0][i]
    if i < 7 or i > 33:
        if "def" in ws_code_line:
            ws_code_line = textwrap.dedent(ws_code_line)
            if "self" not in ws_code_line:
                ws_code_line = ws_code_line.replace("\n", "self,\n")
        ws_base_code.append(ws_code_line)        
        if "return" in ws_code_line:
            break
# ws_base_code = "".join(ws_base_code)
# ws_base_code = str(ws_base_code).replace("[", "")
# ws_base_code = str(ws_base_code).replace("]", "")
# ws_base_code = ws_base_code[0: -1]
ws_base_header = ws_base_alg[0][1:7]
ws_base_return = ws_base_alg[0][-1:]
ws_base_code

In [ ]:
import numpy as np
import json

output_directory = "D:\\projects\\nmf_py\\funsearch\\algorithms"

def run_algorithm(new_algorithm):
    input_file = os.path.join("data", "Dataset-BatonRouge-con.csv")
    uncertainty_file = os.path.join("data", "Dataset-BatonRouge-unc.csv")
    
    data_handler = DataHandler(
        input_path=input_file,
        uncertainty_path=uncertainty_file,
        index_col='Date'
    )
    V, U = data_handler.get_data()
    # TODO: In python code will change this to parallel=True (issues with Jupyter notebooks parallel method)
    nmf_models = BatchNMF(V=V, U=U, factors=6, models=10, method='ls-nmf', parallel=False, verbose=True)
    nmf_models.update_step = new_algorithm
    nmf_models.train()
    return nmf_models

def aggregate_results(models: BatchNMF):
    runtime = models.runtime / models.models
    qtrue = []
    qrobust = []
    for model in models.results:
        if model is None:
            continue
        qtrue.append(model.Qtrue)
        qrobust.append(model.Qrobust)
    return {"runtime": (round(runtime,2), round(models.runtime)), 
            "Q(true)": (round(np.min(qtrue),2), round(np.mean(qtrue),2), round(np.max(qtrue),2)), 
            "Q(robust)": (round(np.min(qrobust),2), round(np.mean(qrobust),2), round(np.max(qrobust),2))}

def update_summary(name, alg_summary, code_path):
    summary_file = os.path.join(output_directory, "summary.json")
    alg_summary["code_path"] = code_path
    if os.path.exists(summary_file):
        alg_key = name
        summary = {name: alg_summary}
        with open(summary_file, 'r') as sum_file:
            existing_summary = json.load(sum_file)
            if alg_key not in existing_summary.keys():
                existing_summary[alg_key] = alg_summary[alg_key]
            alg_summary = existing_summary
    else:
        summary = {name: alg_summary}
        alg_summary = summary
    with open(summary_file, "w") as sum_file:
        json.dump(alg_summary, sum_file)

def save_algorithm(name, code):
    code_file = os.path.join(output_directory, f"gemini-{name}.text")
    with open(code_file, "w") as cfile:
        for cline in code:
            cfile.write(cline)
    return code_file

def select_algorithm():
    summary_file = os.path.join(output_directory, "summary.json")
    index = 1
    if os.path.exists(summary_file):
        code_path = None
        alg_code = None
        with open(summary_file, 'r') as sum_file:
            existing_models = json.load(sum_file)
            model_keys = list(existing_models.keys())
            random_key = np.random.choice(model_keys,1)
            index = len(model_keys)
            while index in model_keys:
                index += 1
            code_path = existing_models[random_key]["code_path"]
        with open(code_path, 'r') as code_file:
            alg_code = code_file.read()
        return index, alg_code
    return index, None

In [ ]:
base_models = run_algorithm(new_algorithm=ws_base_code)

In [ ]:
base_Qrobust = base_models.results[base_models.best_model].Qrobust
base_Qrobust

In [ ]:
base_results = aggregate_results(models=base_models)
code_path = save_algorithm("0", base_code)
update_summary(0, base_results, code_path)

In [ ]:
# STEPS
# gemini_search = True
# n_algs = 4
# alg_keep = 2
# n_algs = 0
# added_algs = 0
# best_q = base_Qrobust
# best_alg = 0

# max_search = 500
# search_i = 0

# while gemini_search:
#     # Select a random existing algorithm
#     index, alg = select_algorithm()
#     if alg is None:
#         alg = base_code
#     # TODO: generate prompt
#     # TODO: parse prompt for model(s)
#     new_algs = []
#     alg_results = []
#     # for each model
#     for new_alg in new_algs:
#         try:
#             new_alg_result = run_algorithm(new_alg)
#         except Exception as e:
#             print(f"Algorithm failed due to error: {e}")
#         alg_qrobust = new_alg_result.results[new_alg_result.best_model].Qrobust
#         alg_results.append((new_alg, alg_qrobust, new_alg_result))
#         n_algs += 1
#     alg_results.sort(key=lambda a: a[2])
#     for i, alg_result in enumerate(alg_results): 
#         if i >= 2:
#             continue
#         if alg_result[1] > 2*base_Qrobust:
#             print(f"Algorithm failed due to Q(robust) being greater than 2*base_Qrobust. Model Q(Robust): {alg_result[1]}")
#             continue
#         elif alg_result[1] < base_Qrobust:
#             best_q = base_Qrobust
#             best_alg = index
#         # if the model succeeded, aggregate the results with 
#         results = aggregate_results(models=alg_result[2])
#         # write the model code to file
#         alg_file = save_algorithm(index, alg_result[0])
#         # write the results to the summary
#         update_summary(index, results, alg_file)
#         index += 1
#         added_algs
#     print(f"Search: {search_i}/{max_search}, Algorithms tested: {n_algs}, Algorithms added: {add_algs}, Current index: {index}, Base Q(robust): {base_Qrobust}, Best Q(robust): {best_q}, Best Algorithm: {best_alg}")
#     search_i += 1
#     if search_i > max_search:
#         gemini_search = False

In [ ]:
GOOGLE_API_KEY = os.getenv("GOOGLE_GEMINI_KEY")
genai.configure(api_key=GOOGLE_API_KEY)
model = genai.GenerativeModel('gemini-pro')

In [ ]:
chat = model.start_chat(history=[])

In [ ]:
test_prompt = f"Can you give me an optimized version of the Non-Negative Matrix Factorization algorithm with input weights (We) using the provided input parameters and outputs in Python only using numpy, and return W and H? Function must be called update and take self as the first argument. The inputs have dimensions V: (NxM), We: (NxM), W: (Nxk) and H: (kxM). Here is the original to work from: {base_code}"

In [ ]:
response = chat.send_message(test_prompt)

In [ ]:
test = str(chat.history)
test

In [ ]:
new_alg = response.text
new_alg

In [ ]:
def parse_response(new_alg_str):
    alg_list = (new_alg_str).split("\n")
    new_alg = []
    copy = False
    for code_line in alg_list:
        if "def" in code_line:
            copy = True
        if copy:
            new_alg.append(code_line)
        if "return" in code_line:
            break
    return "\n".join(new_alg)

In [ ]:
new_code = parse_response(new_alg_str=new_alg)
new_code

In [ ]:
run_algorithm(new_code)

In [ ]:
prompt1 = "Get really creative with the algorithm and try something novel? But don't change the signature"
response1 = chat.send_message(prompt1)

In [ ]:
new_code1 = parse_response(new_alg_str=response1.text)
new_code1

In [ ]:
run_algorithm(new_code1)

In [ ]:
prompt2 = "Give me a more creative algorithm"
response2 = chat.send_message(prompt2)

In [ ]:
new_code2 = parse_response(new_alg_str=response2.text)
response2.text

In [ ]:
run_algorithm(new_code2)

In [ ]:
response_test = chat.send_message("Give me 20 prompts I can submit to you for optimizing, updating, being creative with a snippet of python code for matrix factorization.")
print(response_test.text)

In [ ]:
prompts = response_test.text.split("\n")
prompts